In [ ]:
import csv
import struct # распаковывает координаты из бинарного формата WKB
from pathlib import Path # удобно задаёт и открывает пути к CSV-файлам

In [1]:
from NDVI import init_ee, get_location_score

In [ ]:
INPUT = Path("new_buildings_.csv")
OUTPUT = Path("new_buildings_with_ndvi.csv")

In [ ]:
def parse_wkb_point(hex_wkb: str) -> tuple[float, float]:
    raw = bytes.fromhex(hex_wkb) 
    # WKB Point: byte order + geometry type + X/Y.
    # X = lon, Y = lat.
    lon, lat = struct.unpack("<dd", raw[5:21])
    return lat, lon

In [ ]:
def main() -> None: # основная функция, не возвращает ничего
    init_ee() # инициализирует Google Earth Engine
    with INPUT.open(newline="", encoding="utf-8") as src: # открывает исходный файл
        reader = csv.DictReader(src) # читает CSV-файл как словарь
        fieldnames = list(reader.fieldnames or []) # получает список имен столбцов
        if "coordinates" not in fieldnames: # если в списке нет столбца "coordinates"
            fieldnames.append("coordinates") # добавляет столбец с координатами в формате POINT(lon lat)
        if "ndvi" not in fieldnames: # если в списке нет столбца "ndvi"
            fieldnames.append("ndvi") # добавляет столбец "ndvi"
        rows = []  # создаёт список для хранения строк
        cache: dict[str, float] = {} # создаёт словарь для кэширования результатов
        for i, row in enumerate(reader, start=1): # проходит по всем строкам CSV-файла
            location = row["location"] # получает координату из столбца "location"
            try:
                lat, lon = parse_wkb_point(location)
                row["coordinates"] = f"POINT({lon} {lat})"

                if location not in cache:
                    cache[location] = get_location_score(lat, lon)
                row["ndvi"] = cache[location]
                print(f"{i}: {row['address']} -> {row['coordinates']} -> NDVI {row['ndvi']}")
            except Exception as exc:
                row["coordinates"] = ""
                row["ndvi"] = ""
                print(f"{i}: ошибка для {row['address']}: {exc}")
            rows.append(row)
    with OUTPUT.open("w", newline="", encoding="utf-8") as dst:
        writer = csv.DictWriter(dst, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
if __name__ == "__main__":
    main()